In [1]:
# Load env variables
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from utils.chat import chat, add_assistant_message, add_user_message
from anthropic.types import MessageParam

import json

In [3]:
def run_prompt(test_case):
    """Merges the prompt and test case input then returns the result"""
    prompt = f"""
Please, solve the following task:

{test_case["task"]}

* Respond only with Python, JSON or a plain Regex
* o not add any comments or commentary explanation
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, model="claude-haiku-4-5", stop_sequences=["```"])
    return output

In [4]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution criteria:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, model="claude-sonnet-4-5", stop_sequences=["```"])
    return json.loads(eval_text)


In [5]:
import re
import ast

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0
    
def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0
    
def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0
    
def grade_syntax(test_case, response):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [6]:
from rich import print

def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    model_reasoning = model_grade["reasoning"]
    print(model_grade)

    syntax_score = grade_syntax(test_case, output)

    return {
        "output": output,
        "test_case": test_case,
        "score": (model_score + syntax_score) / 2,
        "reasoning": model_reasoning
    }

In [7]:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results =  [run_test_case(test_case) for test_case in dataset]

    avg_score = mean(result["score"] for result in results)
    print(f"AVG score: {avg_score}")

    return results    

In [8]:
import json

with open("002_prompts/dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

FileNotFoundError: [Errno 2] No such file or directory: '002_prompts/dataset.json'

In [ ]:
from rich import print

print(results)